# 01 · Data understanding (Unit I)

Python essentials, NumPy, pandas, data types, missing values, duplicates, encoding, scaling and the train/validation/test split.

Data: UCI Bike Sharing `hour.csv` (official source, CC BY 4.0).

In [ ]:
# Make the project folder importable (notebooks live in notebooks/)
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import Image, display
from config import project_config as cfg
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Load and validate the real dataset
`load_hour_data()` stops with an error if the file does not match the official UCI format.

In [ ]:
from src.data_loader import load_hour_data, summarise_data
df = load_hour_data()
print(df.shape)
df.head()

## 2. Data types and statistical summary

In [ ]:
df.info()

In [ ]:
df[cfg.HOUR_COLUMNS].describe().T.round(3)

## 3. NumPy: arrays and vectorised operations
UCI stores temperature on a 0–1 scale. With NumPy we convert the whole column to °C in one operation (no loop).

In [ ]:
cnt = df['cnt'].to_numpy()          # pandas column -> NumPy array
print(type(cnt), cnt.shape, cnt.dtype)
print('mean', cnt.mean().round(2), '| std', cnt.std().round(2),
      '| 25/50/75th percentiles', np.percentile(cnt, [25, 50, 75]))
temp_c = df['temp'].to_numpy() * (cfg.TEMP_MAX_C - cfg.TEMP_MIN_C) + cfg.TEMP_MIN_C
print('temperature range in °C:', temp_c.min().round(1), 'to', temp_c.max().round(1))
print('hours with more than 500 rentals:', np.sum(cnt > 500))

## 4. Missing values, duplicates and missing hours

In [ ]:
print('Missing values per column:')
print(df[cfg.HOUR_COLUMNS].isna().sum().to_string())
summary = summarise_data(df)
{k: summary[k] for k in ['rows', 'missing_values_total', 'duplicate_rows', 'duplicate_timestamps', 'hours_in_date_range', 'missing_hours']}

**Finding:** the file has **no missing cell values** and **no duplicate rows**, so no imputation or de-duplication is needed. We do **not** invent values.

Separately, 165 hours have **no row at all** (for example the Hurricane Sandy shutdown). These are not filled in, because inventing rental counts would be fabricating data.

**Strategy if missing values appeared** (for example in user input): reject invalid input (`src/predict.py`), or put a `SimpleImputer` *inside* the pipeline so it is fitted on training data only. The check below confirms the imputer would change nothing here.

In [ ]:
from sklearn.impute import SimpleImputer
num = df[['temp', 'hum', 'windspeed']]
imputed = SimpleImputer(strategy='median').fit_transform(num)
print('Values changed by imputation:', int((imputed != num.to_numpy()).sum()))

## 5. Categorical encoding
`season`, `weathersit`, `hr` etc. are **codes**, not quantities (season 4 is not 'twice' season 2). Linear models therefore get **one-hot** columns.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder(sparse_output=False).fit(df[['season']])
pd.DataFrame(enc.transform(df[['season']].head(3)), columns=enc.get_feature_names_out())

## 6. Chronological train / validation / test split
Random shuffling would let the model see the future (demand grew ~63% from 2011 to 2012), so we split by date.

In [ ]:
from src.feature_engineering import add_features
from src.preprocessing import chronological_split, split_summary
train, val, test = chronological_split(add_features(df))
split_summary(train, val, test)

## 7. Feature scaling (fitted on training data only)
`StandardScaler` learns mean and standard deviation **from the training period**, then applies them unchanged to validation and test.

In [ ]:
from sklearn.preprocessing import StandardScaler
cols = ['temp', 'hum', 'windspeed']
scaler = StandardScaler().fit(train[cols])
print('means learned from train:', scaler.mean_.round(3))
print('validation after scaling -> mean', scaler.transform(val[cols]).mean(axis=0).round(2),
      '(not exactly 0, because the scaler never saw validation data)')